# Medical Abstract Classification
**Course:** SSIM916 — Problem Set #2: Using Text as Data  
**Student Number:** 750091800  
**Date:** 26 March 2026


## Section 0: Configuration and Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os, json
from tqdm import tqdm
from IPython.display import display

plt.style.use('ggplot')
np.random.seed(42)

CONFIG = {
    'data_dir': 'data',
    'results_dir': 'results',
    'random_state': 42
}
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print('Setup complete.')


## Section 1: Research Question
> **Research Question:** Can machine learning models trained on weakly-supervised medical abstracts reliably classify them by research type (Diagnosis, Treatment, Prevention), and which textual features emerge as the strongest interpretable predictors?


## Section 2: Data Loading and Preprocessing


In [ ]:
import sys
sys.path.insert(0, '.')
from src.preprocess import load_data, reconstruct_abstracts, map_labels, balance_dataset

df_lines = load_data(CONFIG['data_dir'])
print(f'Lines loaded: {len(df_lines)}')

df_abs = reconstruct_abstracts(df_lines)
print(f'Abstracts reconstructed: {len(df_abs)}')

df_labeled = map_labels(df_abs)
print('Label distribution (before balancing):')
print(df_labeled['label'].value_counts())


In [ ]:
df_balanced = balance_dataset(df_labeled, target_size=20_000, random_state=42)
print('\nLabel distribution (after stratified undersampling):')
print(df_balanced['label'].value_counts())
display(df_balanced.head(3))


## Section 3: Class Distribution Visualisation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

before = df_labeled['label'].value_counts()
after  = df_balanced['label'].value_counts()

before.plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868'])
axes[0].set_title('Class Distribution — Before Balancing')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

after.plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452','#55A868'])
axes[1].set_title('Class Distribution — After Stratified Undersampling')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

fig.tight_layout()
fig.savefig(os.path.join(CONFIG['results_dir'], 'class_distribution.png'), dpi=300)
plt.show()


## Section 4: Exploratory Visualisations


In [ ]:
# Word clouds per class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, label in enumerate(['Treatment', 'Diagnosis', 'Prevention']):
    subset = df_balanced[df_balanced['label'] == label]['abstract_text']
    text = ' '.join(subset.sample(min(500, len(subset)), random_state=42))
    wc = WordCloud(width=400, height=400, background_color='white',
                   colormap='coolwarm').generate(text)
    axes[i].imshow(wc)
    axes[i].set_title(label, fontsize=14)
    axes[i].axis('off')
fig.suptitle('Top Terms per Class (Word Clouds)', fontsize=16)
fig.tight_layout()
fig.savefig(os.path.join(CONFIG['results_dir'], 'wordclouds.png'), dpi=300)
plt.show()


In [ ]:
# Abstract length distribution
df_balanced['word_count'] = df_balanced['abstract_text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(10, 5))
for label, grp in df_balanced.groupby('label'):
    grp['word_count'].plot.kde(ax=ax, label=label, lw=2)
ax.set_xlabel('Word Count')
ax.set_title('Abstract Length Distribution by Class')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(CONFIG['results_dir'], 'abstract_length_dist.png'), dpi=300)
plt.show()


## Section 5: Train / Test Split


In [ ]:
from sklearn.preprocessing import LabelEncoder
from src.baseline_model import load_split_data

X_train, X_test, y_train, y_test = load_split_data(
    data_dir=CONFIG['data_dir'],
    target_size=20_000,
    random_state=42
)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
classes = le.classes_.tolist()

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Classes: {classes}')


## Section 6: Model 1 — Logistic Regression (C = 10.0)


In [ ]:
from src.baseline_model import train_logreg
from src.evaluate import evaluate_model

mod_lr = train_logreg(X_train, y_train_enc, le, C=10.0)
y_pred_lr = le.inverse_transform(mod_lr.predict(X_test))
y_prob_lr = mod_lr.predict_proba(X_test)

metrics_lr = evaluate_model(
    y_test, y_pred_lr, y_prob_lr,
    classes=classes,
    model_name='Logistic Regression',
    results_dir=CONFIG['results_dir']
)


## Section 7: Model 2 — Multinomial Naive Bayes (alpha = 0.05)


In [ ]:
from src.baseline_model import train_nb

mod_nb = train_nb(X_train, y_train_enc, le, alpha=0.05)
y_pred_nb = le.inverse_transform(mod_nb.predict(X_test))
y_prob_nb = mod_nb.predict_proba(X_test)

metrics_nb = evaluate_model(
    y_test, y_pred_nb, y_prob_nb,
    classes=classes,
    model_name='Multinomial NB',
    results_dir=CONFIG['results_dir']
)


## Section 8: Feature Interpretability (Logistic Regression)


In [ ]:
import numpy as np

tfidf = mod_lr.named_steps['tfidf']
clf   = mod_lr.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, (cls, ax) in enumerate(zip(classes, axes)):
    coefs = clf.coef_[i]
    top_idx = np.argsort(coefs)[-15:][::-1]
    ax.barh(feature_names[top_idx][::-1], coefs[top_idx][::-1], color='#4C72B0')
    ax.set_title(f'Top 15 Features: {cls}', fontsize=13)
    ax.set_xlabel('Coefficient')
fig.tight_layout()
fig.savefig(os.path.join(CONFIG['results_dir'], 'feature_importance_logreg.png'), dpi=300)
plt.show()


## Section 9: Model Comparison Table


In [ ]:
biobert = {
    'model': 'BioBERT v1.2 (benchmark)',
    'accuracy': 93.67,
    'macro_f1': 0.9362,
    'precision': 0.9365,
    'recall': 0.9360,
    'roc_auc': None
}

results_df = pd.DataFrame([metrics_lr, metrics_nb, biobert])
results_df = results_df.set_index('model')
display(results_df)

print('\nAll output figures saved to:', CONFIG['results_dir'])
